# Info

This script performs a Random Forest classification.

The following parameters can be specified:

#### TRAIN_DATA:

The dataset the random forest will be trained on. \

N22 = Northern Bavaria 2022\
N23 = Northern Bavaria 2023\
S20 = Southern Bavaria 2020\
S22 = Southern Bavaria 2022\
ALL = all of the above\

#### VAL_DATA:

The dataset the random forest will be validated on.\

N22 = Northern Bavaria 2022\
N23 = Northern Bavaria 2023\
S20 = Southern Bavaria 2020\
S22 = Southern Bavaria 2022\
ALL = all of the above\

_If VAL_DATA == TRAIN_DATA, as 5-fold Cross Validation will be used._

#### method:

This is the sampling method:\

weighted: Solely minority class ("deadwood") gets oversampled. Specify **weight** as well. Specify **total_n** (number of observations for the model) as well.\
undersampling: All the classes get undersampled to the number of the minority class.\
oversampling: All the classes get oversampled to the number of the majority class ("undisturbed"). Specify, XXXXx as well.




# 1 Import packages and functions

In [24]:
import xarray as xr
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold

In [25]:
from plot_functions import *
from read_functions import *
from datahandling_functions import *
from analysis_functions import *

# 2 Read

In [26]:
chosen_bands = ["blue", "green", "red", "rededge1", "rededge3", "nir", "swir1", "swir2", "ndvi", "savi", "ndwi", "bsi", "contrast"] #options: "blue", "green", "red", "rededge1", "rededge2", "rededge3", "nir", "nir_narrow", "swir1", "swir2", "ndvi", "savi", "ndwi", "bsi", "nssi", "contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"
TRAIN_DATA = "ALL"
VAL_DATA = "ALL"
method = "weighted"
# if method = weight:
weight = 4
FIT = "d"

In [27]:
print(df.head())

                    x             y    blue   green     red  rededge1  \
226581   4.415151e+06  3.022367e+06  0.1203  0.1326  0.1197    0.1546   
1036239  4.598914e+06  2.837897e+06  0.1256  0.1382  0.1207    0.1645   
291755   4.420401e+06  3.021407e+06  0.1196  0.1320  0.1228    0.1687   
648633   4.592264e+06  2.844757e+06  0.1278  0.1417  0.1222    0.1714   
1074235  4.597894e+06  2.836807e+06  0.1201  0.1299  0.1175    0.1448   

         rededge3     nir   swir1   swir2      ndvi      savi      ndwi  \
226581     0.2944  0.3012  0.1784  0.1333  0.431219  0.295635  0.256047   
1036239    0.3994  0.3712  0.2423  0.1654  0.509250  0.378818  0.210106   
291755     0.2775  0.2686  0.2371  0.1773  0.372509  0.245344  0.062290   
648633     0.3922  0.3566  0.2640  0.1775  0.489557  0.359215  0.149210   
1074235    0.2694  0.2807  0.1831  0.1401  0.409844  0.272545  0.210436   

              bsi   contrast  trainclass region region_class  
226581  -0.249815   0.396875           3    n22

## 2.1 Read Train

In [28]:
if TRAIN_DATA == "ALL":

    df1 = open_to_pd_df_withregionlabels(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\n22.tif")
    df2 = open_to_pd_df_withregionlabels(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\n23.tif")
    df3 = open_to_pd_df_withregionlabels(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\s20.tif")
    df4 = open_to_pd_df_withregionlabels(r"C:\Users\miles\OneDrive\Dokumente\ROOT\trainingdata_collection\trainingdata_withindices\s22.tif")
    df = pd.concat([df1, df2, df3, df4], axis=0, ignore_index=True)
    df = df.reset_index()

if TRAIN_DATA != "ALL":

    df = open_to_pd_df(f"C:/Users/miles/OneDrive/Dokumente/ROOT/trainingdata_collection/trainingdata_withindices/{TRAIN_DATA.lower()}.tif")

df.columns.name = None
df = df.loc[:, ~df.columns.duplicated()]
final_selection = ["x", "y"] + [b for b in chosen_bands if b not in ["x", "y", "trainclass"]] + ["trainclass", "region", "region_class"]
train_df = df[final_selection]

## 2.2 Read Val


In [29]:
if VAL_DATA != TRAIN_DATA:

    val_df = open_to_pd_df(f"C:/Users/miles/OneDrive/Dokumente/ROOT/trainingdata_collection/trainingdata_withindices/{VAL_DATA.lower()}.tif")
    val_df = val_df.reset_index()
    val_df.columns.name = None
    val_df = val_df.loc[:, ~val_df.columns.duplicated()]
    final_selection = ["x", "y"] + [b for b in chosen_bands if b not in ["x", "y", "trainclass"]] + ["trainclass"]
    val_df = val_df[final_selection]

    forestclass_test = val_df["trainclass"]
    pred_test = val_df.drop(columns=["trainclass", "region", "regionclass"])

    forestclass_train = train_df["trainclass"]
    pred_train = train_df.drop(columns=["trainclass", "region", "regionclass"])


if VAL_DATA == TRAIN_DATA:

    df = rf_sample(train_df, method = method, weight = weight)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    pred = df.drop(columns=['trainclass', 'region', 'region_class'])
    forestclass = df['trainclass']
    strat_col = df['region_class']

# 3. TRAIN

In [ ]:
if VAL_DATA == TRAIN_DATA:

    all_accuracies = []
    all_f1s = []
    all_cms = []
    cv_predictions = np.zeros_like(forestclass)

    print(f"{'Fold':<10} | {'Accuracy':<10} | {'F1 (Weighted)':<15}")
    print("-" * 45)

    for i, (train_idx, test_idx) in enumerate(skf.split(pred, strat_col)):

        pred_train, pred_test = pred.iloc[train_idx], pred.iloc[test_idx]
        forestclass_train, forestclass_test = forestclass.iloc[train_idx], forestclass.iloc[test_idx]

        rf = randomForestClass(ntrees = 750, pred_train=pred_train, forestclass_train=forestclass_train, FIT = FIT)
        predictions = rf.predict(pred_test)
        cv_predictions[test_idx] = predictions
        acc = accuracy_score(forestclass_test, predictions)

if VAL_DATA != TRAIN_DATA:

    rf = randomForestClass(ntrees = 750, pred_train=pred_train, forestclass_train=forestclass_train)
    predictions = rf.predict(pred_test)

Fold       | Accuracy   | F1 (Weighted)  
---------------------------------------------
unbalanced scikit learn mode was used.


C:\Users\miles\OneDrive\Dokumente\ROOT\jupyter\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


unbalanced scikit learn mode was used.
unbalanced scikit learn mode was used.


# 4. PLOT

In [ ]:
if VAL_DATA == TRAIN_DATA:

    importances = pd.Series(rf.feature_importances_, index=X.columns)

    fig = plot_cross_model_results(
        y_true=forestclass_test,
        y_pred=cv_predictions,
        feature_importances=importances,
        train_mode=TRAIN_DATA,
        val_mode=VAL_DATA,
        class_names=['Disturbed', 'Deadwood', 'Undisturbed'])

    plt.show()

else:

    fig = plot_model_results(
        y_true=forestclass_test,
        y_pred=predictions,
        rf_model=rf,
        feature_names=chosen_bands,
        train_mode=TRAIN_DATA,
        val_mode=VAL_DATA,)
    plt.show()